# PCA pipeline on VDS using VBI sites (k=5 PCs, no pruning)

This notebook:
- Filters on **locus** (broad keep) using `ht_vbi`
- Splits multiallelics at the **VDS** level (`hl.vds.split_multi`)
- Converts to a **full MatrixTable** (all entry fields)
- Refines by **(locus, alleles)** against `ht_vbi`
- Runs **PCA** with `k=5` (no LD-pruning)
- Annotates PCs back onto the MT columns (keeps everything in Hail)

**Reference:** GRCh38


In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1756884101514_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-86-239.ap-southeast-1.compute.internal:37187
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
# from pprint import pprint
# pprint(dict(hl.spark_context().getConf().getAll()))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# =============================================================================
# source / inputs / outputs
# =============================================================================

# ------------------- source ---------------------------------------------------
vds_prefix = "s3://precise-scratch/goypav/1KG/VDS/"

# ------------------- inputs ---------------------------------------------------
vds_uri    = "s3://precise-trust/opendata-1000genomes/1000genomes-vds-n3205.vds"
ht_vbi_uri = "s3://npm-grids/resources/1000g.phase3.100k.b38.fixed.ht/"   # your BED→HT keyed by ('locus','alleles')

# ------------------- outputs --------------------------------------------------
workdir                         = vds_prefix + "pca_work/"

# intermediate checkpoints
vds_locus_uri                   = workdir + "1000genomes.n3205.locus_filtered.vds"
vds_split_multi_uri             = workdir + "1000genomes.n3205.splitmulti.vds"
mt_full_from_vds_split_uri      = workdir + "1000genomes.n3205.splitmulti.full.mt"
mt_vbi_biallelic_uri            = workdir + "1000genomes.n3205.splitmulti.vbi.mt"
mt_with_pcs_uri                 = workdir + "1000genomes.n3205.splitmulti.vbi.pca.mt"

overwrite = False

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [7]:
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

151393

In [8]:
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64

In [9]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 3205
Number of variant partitions: 5767
Total number of variants: 156,228,032

In [10]:
ht_vbi = hl.read_table(ht_vbi_uri)
print(f"    ht_vbi rows (biallelic targets): {ht_vbi.count():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

    ht_vbi rows (biallelic targets): 100,000

## PCA workflow

In [11]:
print(">>> Step 1/5: filtering VDS by locus ...")

ht_vbi_locus = ht_vbi.key_by("locus")
vds_locus = hl.vds.filter_variants(vds, ht_vbi_locus, keep=True)
vds_locus.write(vds_locus_uri, overwrite=overwrite)

# RELOAD from disk
vds_locus = hl.vds.read_vds(vds_locus_uri)

print(f"    wrote & reloaded locus-filtered VDS: {vds_locus_uri}")
print(f"    variant_data rows: {vds_locus.variant_data.count_rows():,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

>>> Step 1/5: filtering VDS by locus ...
    wrote & reloaded locus-filtered VDS: s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.locus_filtered.vds
    variant_data rows: 99,981
2025-09-03 09:52:46.982 Hail: INFO: wrote matrix table with 2887027773 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.locus_filtered.vds/reference_data
2025-09-03 09:52:50.442 Hail: INFO: wrote table with 100000 rows in 1 partition to /tmp/__iruid_69-ulucIKFhlJyh8VqyZ9C7Zk
2025-09-03 09:57:22.127 Hail: INFO: wrote matrix table with 99981 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.locus_filtered.vds/variant_data

In [12]:
print(">>> Step 2/5: splitting multiallelic sites at VDS level ...")

vds_split = hl.vds.split_multi(vds_locus)
vds_split.write(vds_split_multi_uri, overwrite=overwrite)

# RELOAD from disk
vds_split = hl.vds.read_vds(vds_split_multi_uri)

print(f"    wrote & reloaded split VDS: {vds_split_multi_uri}")
print(f"    variant_data rows (post-split): {vds_split.variant_data.count_rows():,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

>>> Step 2/5: splitting multiallelic sites at VDS level ...
    wrote & reloaded split VDS: s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vds
    variant_data rows (post-split): 140,773
2025-09-03 11:08:17.891 Hail: INFO: wrote matrix table with 2887027773 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vds/reference_data
2025-09-03 11:09:40.656 Hail: INFO: wrote matrix table with 140773 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vds/variant_data

In [13]:
print(">>> Step 3/5: converting split VDS to full MT ...")

mt_full = hl.vds.to_dense_mt(vds_split)
mt_full.write(mt_full_from_vds_split_uri, overwrite=overwrite)

# RELOAD from disk
mt_full = hl.read_matrix_table(mt_full_from_vds_split_uri)

print(f"    wrote full MT: {mt_full_from_vds_split_uri}")
print(f"    n_samples:  {mt_full.count_cols():,}")
print(f"    n_variants: {mt_full.count_rows():,}")
print("    schema:")
mt_full.describe()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

>>> Step 3/5: converting split VDS to full MT ...
    wrote full MT: s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.full.mt
    n_samples:  3,205
    n_variants: 140,773
    schema:
----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'a_index': int32
    'was_split': bool
----------------------------------------
Entry fields:
    'ICNT': array<int32>
    'DP': int32
    'SPL': array<int32>
    'GQ': int32
    'MIN_DP': int32
    'GT': call
    'SQ': float64
    'SB': array<int32>
    'F2R1': arra

In [14]:
print(">>> Step 4/5: filtering MT by exact (locus, alleles) in ht_vbi ...")

mt = mt_full.filter_rows(hl.is_defined(ht_vbi[mt_full.row_key]))
mt.write(mt_vbi_biallelic_uri, overwrite=overwrite)

# RELOAD from disk
mt = hl.read_matrix_table(mt_vbi_biallelic_uri)

print(f"    wrote & reloaded refined MT: {mt_vbi_biallelic_uri}")
print(f"    n_samples:  {mt.count_cols():,}")
print(f"    n_variants: {mt.count_rows():,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

>>> Step 4/5: filtering MT by exact (locus, alleles) in ht_vbi ...
    wrote & reloaded refined MT: s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vbi.mt
    n_samples:  3,205
    n_variants: 99,774
2025-09-03 15:23:11.129 Hail: INFO: wrote table with 100000 rows in 1 partition to /tmp/__iruid_1003-STvJvr2YoYypQhwl1ug80w
2025-09-03 15:24:26.921 Hail: INFO: wrote matrix table with 99774 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vbi.mt

In [23]:
# =============================================================================
# Step 5/5: PCA (k=5) on GT/LGT, annotate PCs on columns, eigenvalues in GLOBALS
# =============================================================================
print(">>> Step 5/5: running PCA (k=5) and annotating PCs ...")

# read mt
mt = hl.read_matrix_table(mt_vbi_biallelic_uri)

# run PCA
eigs_list, scores_ht, loadings_ht = hl.hwe_normalized_pca(mt.GT, k=5)

print("    eigenvalues:")
for i, ev in enumerate(eigs_list, 1):
    print(f"      PC{i}: {ev}")

# 1) annotate eigenvalues ON GLOBALS (once)
mt = mt.annotate_globals(pca_eigenvalues = hl.array(eigs_list))

# 2) annotate PCs ON COLUMNS
scores_by_s = scores_ht.select('scores').key_by('s')
mt = mt.annotate_cols(
    pca = hl.struct(
        PC1 = scores_by_s[mt.s].scores[0],
        PC2 = scores_by_s[mt.s].scores[1],
        PC3 = scores_by_s[mt.s].scores[2],
        PC4 = scores_by_s[mt.s].scores[3],
        PC5 = scores_by_s[mt.s].scores[4],
    )
)

# write
mt.write(mt_with_pcs_uri, overwrite=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

>>> Step 5/5: running PCA (k=5) and annotating PCs ...
    eigenvalues:
      PC1: 314.96214859357764
      PC2: 136.29505583490158
      PC3: 39.099493930342675
      PC4: 30.292240365046723
      PC5: 4.934625817191764
2025-09-03 16:03:08.023 Hail: INFO: hwe_normalize: found 99770 variants after filtering out monomorphic sites.
2025-09-03 16:03:16.379 Hail: INFO: pca: running PCA with 5 components...
2025-09-03 16:03:55.412 Hail: INFO: wrote table with 0 rows in 0 partitions to /tmp/persist_Table7aUXWeuVdK
2025-09-03 16:05:27.969 Hail: INFO: wrote matrix table with 99774 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vbi.pca.mt

In [25]:
# load and check
mt = hl.read_matrix_table(mt_with_pcs_uri)

print(f"    wrote & reloaded final MT with PCs: {mt_with_pcs_uri}")
print(f"    n_samples:  {mt.count_cols():,}")
print(f"    n_variants: {mt.count_rows():,}")
print("    globals now include: pca_eigenvalues")
mt.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

    wrote & reloaded final MT with PCs: s3://precise-scratch/goypav/1KG/VDS/pca_work/1000genomes.n3205.splitmulti.vbi.pca.mt
    n_samples:  3,205
    n_variants: 99,774
    globals now include: pca_eigenvalues
----------------------------------------
Global fields:
    'pca_eigenvalues': array<float64>
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
    'pca': struct {
        PC1: float64, 
        PC2: float64, 
        PC3: float64, 
        PC4: float64, 
        PC5: float64
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'a_index': int32
    'was_split': bool
----------------------------------------
Entry fields:
   